In [1]:
from scripts.dataset import X_train, X_test, y_train, y_test, groups_train, groups_test, numeric_features, categorical_features

DATASET
Total samples:       6,881
Total trajectories:  750
TRAIN / TEST SPLIT
Train samples:       5,491
Test samples:        1,390
Train trajectories:  600
Test trajectories:   150
TRAIN LABEL DISTRIBUTION
       count  percentage
label                   
-1      1457       26.53
 0       231        4.21
 1      3803       69.26
TEST LABEL DISTRIBUTION
       count  percentage
label                   
-1       402       28.92
 0        56        4.03
 1       932       67.05
LEAKAGE CHECK
Overlapping trajectories: 0
X / y / groups alignment: OK
Trajectory split:          OK


In [2]:
def build_transformer_frame(X, y):
    df = X.copy()

    df["text"] = (
        "[CONTEXT]\n"
        + df["context_text"].fillna("")
        + "\n\n[CURRENT]\n"
        + df["current_text"].fillna("")
    )

    # Hugging Face classification labels should be 0..num_labels-1
    label_map = {
        -1: 0,
         0: 1,
         1: 2,
    }

    df["labels"] = (
        y.reset_index(drop=True)
        .map(label_map)
        .astype(int)
    )

    return df[["text", "labels"]].reset_index(drop=True)

In [3]:
train_df = build_transformer_frame(
    X_train.reset_index(drop=True),
    y_train.reset_index(drop=True),
)

test_df = build_transformer_frame(
    X_test.reset_index(drop=True),
    y_test.reset_index(drop=True),
)

print(train_df.shape)
print(test_df.shape)

print(train_df["labels"].value_counts())

(5491, 2)
(1390, 2)
labels
2    3803
0    1457
1     231
Name: count, dtype: int64


In [4]:
from datasets import Dataset

hf_train = Dataset.from_pandas(
    train_df,
    preserve_index=False,
)

hf_test = Dataset.from_pandas(
    test_df,
    preserve_index=False,
)

In [5]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [6]:
MAX_LENGTH = 512

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )


tokenized_train = hf_train.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
)

tokenized_test = hf_test.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
)

Map:   0%|          | 0/5491 [00:00<?, ? examples/s]

Map:   0%|          | 0/1390 [00:00<?, ? examples/s]

In [7]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [8]:
from transformers import AutoModelForSequenceClassification

id2label = {
    0: "-1",
    1: "0",
    2: "+1",
}

label2id = {
    "-1": 0,
    "0": 1,
    "+1": 2,
}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1,
    )

    return {
        "accuracy": accuracy_score(
            labels,
            predictions,
        ),

        "macro_f1": f1_score(
            labels,
            predictions,
            average="macro",
        ),

        "error_precision": precision_score(
            labels,
            predictions,
            labels=[0],  # original -1
            average="macro",
            zero_division=0,
        ),

        "error_recall": recall_score(
            labels,
            predictions,
            labels=[0],
            average="macro",
            zero_division=0,
        ),

        "error_f1": f1_score(
            labels,
            predictions,
            labels=[0],
            average="macro",
            zero_division=0,
        ),
    }

In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./distilbert-agent-reliability",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    logging_steps=50,

    seed=42,

    report_to="none",
)

In [12]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,

    processing_class=tokenizer,
    data_collator=data_collator,

    compute_metrics=compute_metrics,
)

In [13]:
trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
transformer_results = trainer.evaluate()

transformer_results

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
)

pred_output = trainer.predict(
    tokenized_test
)

predictions = np.argmax(
    pred_output.predictions,
    axis=-1,
)

labels = pred_output.label_ids

print(
    classification_report(
        labels,
        predictions,
        labels=[0, 1, 2],
        target_names=["-1", "0", "+1"],
        zero_division=0,
        digits=4,
    )
)

print(
    confusion_matrix(
        labels,
        predictions,
        labels=[0, 1, 2],
    )
)

Yes. At this point you have enough experimental evidence to write a solid **Results and Observations** section. I would record it approximately like this.

# Experimental Results: BERT/Transformer Models and Dataset Analysis

## 1. Objective

The goal of the experiments was to determine whether contextual and semantic models can reliably classify individual agent steps into three classes:

* `-1` — erroneous or unreliable step
* `0` — neutral/intermediate class
* `+1` — correct step

Because agent actions inherently depend on previous steps, the experiments also investigated whether trajectory context improves classification and whether models generalize across different agent datasets.

The primary metric was **Macro F1**, rather than accuracy, because the dataset is imbalanced, particularly for class `0`.

---

# 2. Dataset

The combined dataset contains:

| Property               | Value |
| ---------------------- | ----: |
| Samples                | 6,881 |
| Trajectories           |   750 |
| Dataset A trajectories |   250 |
| Dataset B trajectories |   250 |
| Dataset C trajectories |   250 |

For the standard IID experiment, trajectories were separated to prevent leakage:

| Split | Samples | Trajectories |
| ----- | ------: | -----------: |
| Train |   5,491 |          600 |
| Test  |   1,390 |          150 |

No trajectory appeared in both training and test data.

The overall training distribution was strongly imbalanced:

| Label | Train samples | Percentage |
| ----- | ------------: | ---------: |
| `-1`  |         1,457 |     26.53% |
| `0`   |           231 |      4.21% |
| `+1`  |         3,803 |     69.26% |

The very small representation of class `0` proved to be an important difficulty throughout the experiments.

---

# 3. Context-window experiments

Since an agent step may depend on what happened previously in the trajectory, previous steps were incorporated as textual context.

Different history-window sizes were evaluated with grouped cross-validation.

| Context window | Mean Macro F1 |   Std |
| -------------: | ------------: | ----: |
|              1 |         0.586 | 0.031 |
|          **2** |     **0.602** | 0.032 |
|              4 |         0.590 | 0.029 |
|              8 |         0.580 | 0.025 |
|             16 |         0.587 | 0.030 |
|             32 |         0.584 | 0.031 |
|   Full history |         0.584 | 0.032 |

The best result occurred with a context window of **2 previous steps**.

This produced an important observation:

> More context does not necessarily improve reliability classification.

Recent local context was more useful than including long portions of the trajectory. Longer histories may introduce irrelevant information and noise.

---

# 4. Feature ablation

Ablation experiments were performed to determine which information contributed to classification.

| Features               | Mean Macro F1 |
| ---------------------- | ------------: |
| **All features**       |     **0.602** |
| Current + context text |         0.588 |
| Structural + current   |         0.563 |
| Current text only      |         0.541 |
| Structural + context   |         0.516 |
| Context only           |         0.477 |
| Structural only        |         0.411 |

The results indicate that no individual feature family was sufficient.

The strongest model combined:

**current step + recent context + structural trajectory information.**

In particular, context by itself performed poorly (`0.477`), but adding context to current-step information improved performance. This suggests that history is useful primarily for interpreting the current action rather than as an independent predictor.

---

# 5. LinearSVC + TF-IDF

A Linear Support Vector Classifier was trained using TF-IDF representations together with structural and categorical features.

After tuning the regularization parameter, the best value was approximately:

```text
C = 2.0
```

The final IID test performance was:

| Metric   |    Result |
| -------- | --------: |
| Accuracy | **0.845** |
| Macro F1 | **0.655** |

Per-class results:

| Label | Precision | Recall |       F1 |
| ----- | --------: | -----: | -------: |
| `-1`  |      0.78 |   0.76 | **0.77** |
| `0`   |      0.61 |   0.20 | **0.30** |
| `+1`  |      0.87 |   0.92 | **0.90** |

Confusion matrix:

```text
[[304   2  96]
 [ 17  11  28]
 [ 67   5 860]]
```

Grouped cross-validation gave approximately:

```text
Macro F1:     0.614 ± 0.021
Error F1:     0.674 ± 0.045
Error Recall: 0.629 ± 0.037
```

LinearSVC therefore became the strongest IID model tested.

---

# 6. Frozen embedding models

The next experiment investigated whether pretrained semantic embeddings would outperform sparse TF-IDF representations.

Two variants were tested:

| Model                            |  Accuracy |  Macro F1 |  Error F1 |
| -------------------------------- | --------: | --------: | --------: |
| Embeddings only                  |     0.768 |     0.519 |     0.609 |
| Embeddings + structural features | **0.805** | **0.551** | **0.686** |

Adding structural information substantially improved the embedding model, but the resulting Macro F1 of `0.551` remained below LinearSVC's `0.655`.

Therefore, replacing lexical TF-IDF features with pretrained semantic embeddings was **not sufficient to improve overall classification**.

---

# 7. Fine-tuned DistilBERT

The next stage fine-tuned a pretrained DistilBERT sequence classifier.

The model input contained the current agent step together with recent trajectory context.

Token-length analysis showed:

```text
Median:          189 tokens
90th percentile: 909
95th percentile: 1332
99th percentile: ~2501
Maximum:         4845
```

Approximately **21.3% of examples exceeded the 512-token limit**.

A context-aware truncation strategy was therefore introduced that prioritized the current step and most recent context instead of blindly truncating the complete sequence.

This produced only a small improvement, indicating that truncation was not the main source of error.

---

# 8. Class imbalance and weighted fine-tuning

Initial Transformer experiments performed particularly poorly on class `0`.

Because class `0` accounts for only approximately 4% of the training data, class-weighted cross-entropy loss was introduced.

For the corrected experiment, the original training trajectories were further divided into:

```text
480 trajectories → fine-tuning
120 trajectories → validation
150 trajectories → untouched final test
```

Corresponding sample counts were:

| Split             | Samples |
| ----------------- | ------: |
| Transformer train |   4,435 |
| Validation        |   1,056 |
| Final test        |   1,390 |

The training labels were:

```text
-1: 1196
 0:  187
+1: 3052
```

Class weights were calculated **only from the Transformer training set**.

---

# 9. Corrected DistilBERT validation results

Weighted DistilBERT produced the following validation progression:

| Epoch |  Accuracy |  Macro F1 |  Error F1 |
| ----: | --------: | --------: | --------: |
|     1 |     0.675 |     0.439 |     0.502 |
|     2 |     0.741 |     0.480 |     0.569 |
| **3** | **0.773** | **0.512** | **0.574** |

The best checkpoint according to Macro F1 was epoch 3.

Interestingly, validation loss was lowest around epoch 2 while Macro F1 continued improving at epoch 3. This illustrates that weighted cross-entropy loss and Macro F1 do not necessarily move together.

---

# 10. Final DistilBERT test result

The untouched test set produced:

| Metric   |    Result |
| -------- | --------: |
| Accuracy | **0.774** |
| Macro F1 | **0.555** |

Per-class performance:

| Label | Precision | Recall |        F1 |
| ----- | --------: | -----: | --------: |
| `-1`  |     0.661 |  0.634 | **0.647** |
| `0`   |     0.233 |  0.125 | **0.163** |
| `+1`  |     0.836 |  0.873 | **0.854** |

Confusion matrix:

```text
[[255  20 127]
 [ 16   7  33]
 [115   3 814]]
```

The Transformer remained particularly weak on class `0`, correctly detecting only **7 of 56** examples.

The final IID comparison therefore favored LinearSVC:

| Model                   |  Accuracy |  Macro F1 |
| ----------------------- | --------: | --------: |
| **LinearSVC + TF-IDF**  | **0.845** | **0.655** |
| Weighted DistilBERT     |     0.774 |     0.555 |
| Embeddings + structural |     0.805 |     0.551 |
| Embeddings only         |     0.768 |     0.519 |

A larger and more computationally expensive neural model therefore did **not** automatically produce better reliability classification.

---

# 11. Cross-dataset generalization

The models were subsequently evaluated using leave-one-dataset-out experiments:

```text
A + B → C
A + C → B
B + C → A
```

This tests a substantially harder question: whether a reliability classifier trained on some agent environments can generalize to a previously unseen dataset.

For weighted DistilBERT:

| Experiment | Validation Macro F1 | Held-out Macro F1 |
| ---------- | ------------------: | ----------------: |
| A+B → C    |               0.543 |         **0.294** |
| A+C → B    |               0.599 |         **0.287** |
| B+C → A    |               0.575 |         **0.305** |

The average held-out Macro F1 was only approximately:

```text
0.296
```

The drops were substantial:

```text
A+B → C:  -0.249
A+C → B:  -0.312
B+C → A:  -0.270
```

This means DistilBERT learned patterns that worked within the training domains but transferred poorly to a new dataset.

The same general problem was observed with the simpler models:

| Experiment   | LinearSVC | Frozen embeddings | DistilBERT |
| ------------ | --------: | ----------------: | ---------: |
| A+B → C      |     0.300 |         **0.315** |      0.294 |
| A+C → B      |     0.382 |         **0.399** |      0.287 |
| B+C → A      | **0.420** |             0.390 |      0.305 |
| Approx. mean |     0.367 |         **0.368** |      0.296 |

Thus, fine-tuning a Transformer did not solve cross-dataset generalization.

---

# 12. Dataset-shift analysis

Analysis of A, B, and C revealed substantial differences.

### Context and message length

| Statistic             |         A |    B |        C |
| --------------------- | --------: | ---: | -------: |
| Average context words | **488.5** | 91.9 | **28.4** |
| Median context words  |   **345** |   52 |   **14** |
| Average current words |      53.3 | 54.0 | **25.2** |
| Median current words  |        32 |   18 |    **7** |

Dataset A therefore contains dramatically longer contexts than C.

This is a strong form of structural distribution shift.

### Label distributions

| Dataset |      `-1` |      `0` |      `+1` |
| ------- | --------: | -------: | --------: |
| A       |     24.4% | **7.6%** |     68.0% |
| B       | **31.2%** |     3.6% |     65.2% |
| C       |     22.0% |     4.0% | **74.0%** |

The relationship between role and label also differs.

For example, the percentage of tool calls labeled `-1` was:

```text
A: 15.1%
B: 35.1%
C: 16.0%
```

Thus, a model could learn that tool calls are strongly associated with errors in B, but that relationship does not transfer to A or C.

---

# 13. Lexical/domain shift

TF-IDF analysis showed clearly different vocabularies.

Dataset A was characterized by terms such as:

```text
document
search
film
born
school
university
```

Dataset B contained:

```text
order
reservation
price
phone
get_details_by_id
```

Dataset C contained:

```text
startengine
fuel
ignitionmode
fillfueltank
file_name
psi
```

This suggests that the datasets represent substantially different tools, tasks, and environments.

The strongest evidence came from a separate dataset-source classification experiment.

A TF-IDF + LinearSVC model was trained to predict:

```text
A vs B vs C
```

using the trajectory text.

The result was:

> **Dataset-source accuracy = 1.000 ± 0.000**

In other words, the dataset of origin could be identified with **100% cross-validated accuracy from the text**.

This provides quantitative evidence that A, B, and C have extremely strong domain-specific signatures.

---

# 14. Main observation

The experiments suggest that the central limitation is **not simply insufficient model capacity**.

The evidence instead points toward substantial **dataset/domain shift**.

Within a familiar distribution, models can exploit:

* tool names and tool arguments,
* domain-specific vocabulary,
* task-specific language,
* message and context length,
* current role,
* trajectory structure,
* dataset-specific relationships between roles and errors.

This produces relatively strong IID performance, particularly for LinearSVC:

```text
IID Macro F1 ≈ 0.655
```

However, when evaluated on a new dataset, performance falls to approximately:

```text
Macro F1 ≈ 0.29–0.42
```

Even fine-tuned DistilBERT exhibits this collapse.

Therefore, the current evidence supports the hypothesis that the classifiers are learning a combination of **general reliability signals and benchmark-specific shortcuts**, rather than a fully domain-invariant representation of agent reliability.

## Current conclusion

The strongest IID model remains **LinearSVC + TF-IDF with contextual and structural features**.

However, the most important result of the project is arguably not which classifier wins the IID benchmark. It is that **cross-dataset reliability detection remains difficult across all tested model families**.

The 100% dataset-source classification result provides a plausible explanation: the three benchmarks are highly separable in vocabulary, structure, task domain, context length, and label relationships.

The next experimental stage should therefore focus on **domain-invariant reliability features and shortcut reduction**, rather than simply increasing model size.


Yes — **partly**, but there's an important distinction.

Your dataset setup is actually **very useful for evaluating robustness**, precisely because A, B, and C are so different. But I would not conclude that this automatically means it perfectly represents real-world nondeterministic agents.

### What your results demonstrate

You effectively have two kinds of evaluation:

```text
IID evaluation

Train:
A + B + C
   ↓
unseen trajectories from A + B + C
   ↓
LinearSVC Macro F1 ≈ 0.655
```

This asks:

> Can the model generalize to new trajectories drawn from environments similar to those it has already seen?

Your model does reasonably well.

Then you have:

```text
OOD / cross-dataset evaluation

Train: A + B
           ↓
Test:      C

Macro F1 ≈ 0.30
```

This asks a harder and arguably more realistic robustness question:

> Can the reliability detector work in a substantially different agent environment that wasn't represented during training?

And currently the answer is largely **no**.

That's valuable information.

### Why your datasets are useful

Your analysis showed huge differences:

```text
Context length:
A ≈ 488 words
B ≈ 92
C ≈ 28

Vocabulary:
A → documents/search/knowledge tasks
B → orders/reservations/customer tasks
C → engines/files/tools/etc.

Dataset-source prediction:
100%
```

So A/B/C give you meaningful distribution shifts.

That's excellent for testing whether your detector learns something like:

> "This action is inconsistent with what happened previously."

versus something much weaker like:

> "I've seen this tool/domain/text pattern associated with errors before."

Your cross-dataset experiments suggest the models are currently learning a fair amount of the second kind.

---

### But this isn't the same thing as agent nondeterminism

This distinction matters.

**Nondeterminism** means roughly that the same or similar situation can lead an agent to different actions:

```text
same task/context
       ↓
Agent run #1 → action A
Agent run #2 → action B
Agent run #3 → action C
```

Your A/B/C experiment primarily measures **distribution/domain shift**:

```text
Environment A
   ≠
Environment B
   ≠
Environment C
```

Those are related real-world problems, but they aren't identical.

A production reliability detector may face several shifts simultaneously:

```text
new tasks
new tools
new tool schemas
different agents
different LLMs
different prompts
different trajectory lengths
different user behavior
agent nondeterminism
software/environment changes
```

Your cross-dataset experiment captures **some of this diversity**, especially new domains, tools, vocabulary and trajectory structures.

It doesn't prove robustness to all of it.

### In fact, your 100% source-classification result is both good and bad

It's **good for evaluation** because A/B/C provide strong OOD tests.

But it's potentially **bad for training**.

Because:

```text
A/B/C are extremely distinguishable
```

the model can learn:

```text
"This looks like dataset B"
        ↓
"B tool calls have a high error rate"
        ↓
predict -1
```

rather than actually reasoning:

```text
previous state
      +
requested task
      +
current action
      ↓
is this action valid?
```

Your role-label analysis already gives evidence of this problem:

```text
P(error | TOOL_CALL)

A ≈ 15%
B ≈ 35%
C ≈ 16%
```

A classifier can exploit that correlation within B. It fails when that relationship changes.

---

## So I would characterize your dataset this way

Not:

> "The dataset is very reliable because it is exactly like the real world."

Rather:

> **The dataset collection is valuable because its heterogeneous domains allow us to test both in-distribution performance and cross-domain robustness. The large differences between datasets expose shortcut learning that would be hidden by a conventional random train/test split.**

That's a stronger and more defensible conclusion.

Your `0.655` LinearSVC result alone might have led you to believe:

```text
reliability detector works reasonably well ✓
```

But your cross-dataset experiments revealed:

```text
IID
Macro F1 = 0.655
       ↓

OOD
Macro F1 ≈ 0.30–0.42
       ↓

Detector is not robust yet
```

And the source classifier:

```text
A/B/C prediction = 100%
```

gives you a plausible explanation.

### This is actually closer to the problem you ultimately care about

If your long-term goal is a detector that sits alongside arbitrary agents:

```text
                 Reliability detector
                         ↑
                         │
      ┌──────────────────┼──────────────────┐
      │                  │                  │
 shopping agent      coding agent       browser agent
      │                  │                  │
 new tools           new tasks          new domains
```

then you **don't want** a detector that only recognizes patterns from agents it saw during training.

You want something closer to:

```text
trajectory history
       +
current action
       ↓
general consistency / correctness signals
       ↓
reliability prediction
```

Your experiments show you haven't reached that yet.

And that's why I wouldn't view the low cross-dataset scores as simply a failure of your project. **They've identified the actual hard problem your next modeling stage needs to attack: domain-invariant agent reliability detection.**
